# Module 11: Classes, Type Hints, Pydantic, Decorators and Async

**Utrains Python Fundamentals** &middot; lab notebook

*The tools that round out your Python toolkit for AI engineering work.*

## By the end of this notebook you can

- Bundle data and behaviour together in a class, and read dotted SDK output
- Describe a shape with type hints and a dataclass
- Validate untrusted data with a Pydantic model
- Wrap a function with a decorator without changing its code
- Run several slow calls at once with async and await

## How to work through it

Module 11 is the one module with no slide deck, so this notebook follows the course reference guide's section order instead. Sections are numbered rather than labelled with a slide number.

The headings below are the slide numbers from the deck. The explanation for
each one is on the slide and in the [README](../README.md); this notebook is
where you run the code.

Run every cell in order with **Shift + Enter**.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

**Assumed knowledge.** Modules 1 to 10. This is the only notebook that needs a third party package: **Pydantic**, which is in `requirements.txt`.

### Before you start: check Pydantic is available

If this cell fails, your environment is missing Pydantic. Install it with
`uv pip install -r requirements.txt`, then restart the kernel.

In [ ]:
import pydantic

print("pydantic", pydantic.VERSION)

## Section 1 &middot; Classes and objects

A class is a blueprint for a data type that bundles values and behaviour
together. `__init__` sets up what a new object starts with, and `self` refers
to the specific object being worked on.

In [ ]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def describe(self):
        return f"{self.title} by {self.author}"


b = Book("Dune", "Frank Herbert")
print(b.describe())

Wrapping a model API in a class keeps its configuration and its conversation
history bundled together, instead of scattered across separate variables.

In [ ]:
class ChatClient:
    def __init__(self, model):
        self.model = model
        self.history = []

    def ask(self, user_message):
        self.history.append({"role": "user", "content": user_message})
        reply = f"[{self.model}] reply to: {user_message}"
        self.history.append({"role": "assistant", "content": reply})
        return reply


client = ChatClient("claude-sonnet-4-6")
print(client.ask("Hi, I am Serge."))
print(client.ask("What did I just say?"))
print("history length:", len(client.history))

### Reading dotted objects

Now that you know a class holds attributes on `self`, you can read SDK output
the same way: an object, then an attribute, then maybe a list index, then
another attribute. Real SDK responses look exactly like this.

In [ ]:
class FakeDelta:
    def __init__(self, content):
        self.content = content


class FakeChoice:
    def __init__(self, content):
        self.delta = FakeDelta(content)


class FakeChunk:
    def __init__(self, content):
        self.choices = [FakeChoice(content)]


chunk = FakeChunk("Hello Serge!")
print(chunk.choices[0].delta.content)

# A real Anthropic response follows the same shape:  r.content[0].text

---

### Your turn 1

Write a `Deployment` class that starts as pending and can be marked complete. Fill in the constructor's name and the reference to the object itself.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
class Deployment:
    # TODO: name the constructor, and store both values on the object.
    def ____(self, service):
        ____.service = service
        ____.status = "pending"

    def complete(self):
        # TODO: change this object's own status.
        ____.status = "done"


d = Deployment("api-gateway")
print(d.service, "->", d.status)

d.complete()
print(d.service, "->", d.status)

## Section 2 &middot; Type hints and dataclasses

A type hint is a note saying what type a variable or function expects. Python
does not enforce it by itself, but your editor and several AI libraries read it
and catch mistakes early.

A **dataclass** uses type hints to define the exact shape of structured data
without writing a full class by hand. This is how function-calling schemas are
described in code.

In [ ]:
from dataclasses import dataclass


def add(a: int, b: int) -> int:
    return a + b


print(add(2, 3))


@dataclass
class Message:
    role: str
    content: str


@dataclass
class WeatherLookup:
    city: str
    unit: str = "celsius"


print(Message(role="user", content="Hi there"))
print(WeatherLookup(city="Boston"))
print("a dataclass will NOT stop this:", Message(role=123, content=None))

## Section 3 &middot; Pydantic models

A dataclass describes a shape but will not stop you putting the wrong type in a
field, as the last line above showed. **Pydantic** does the same job and
validates every field automatically, raising a clear error the moment something
does not match.

This is the standard way to describe a structured response you expect back from
an AI model: instead of trusting that the model's JSON is shaped correctly, you
validate it on the way in.

In [ ]:
from pydantic import BaseModel, ValidationError


class Person(BaseModel):
    name: str
    age: int


p = Person(name="Alice", age=25)
print(p.name, p.age)


class MovieRecommendation(BaseModel):
    title: str
    year: int
    reason: str


raw_response = '{"title": "Arrival", "year": 2016, "reason": "Language and time."}'

recommendation = MovieRecommendation.model_validate_json(raw_response)
print(recommendation.title, recommendation.year)

In [ ]:
bad_response = '{"title": "Arrival", "year": "not-a-year", "reason": "..."}'

try:
    MovieRecommendation.model_validate_json(bad_response)
except ValidationError as e:
    print("The model returned something unexpected:")
    print(e)

## Section 4 &middot; Decorators

A decorator is a function that wraps another function to add behaviour, without
changing the code inside that function. It is written as an `@` placed directly
above a `def` line. It builds on Module 8's idea that a function is just a
value you can pass around.

In [ ]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        return func(*args, **kwargs)
    return wrapper


@announce
def greet(name):
    return f"Hello, {name}!"


print(greet("Alice"))

In [ ]:
import time


def timed(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.2f}s")
        return result
    return wrapper


@timed
def call_model(prompt):
    time.sleep(0.3)          # pretend this is a network call
    return f"response to: {prompt}"


print(call_model("Summarize this document."))

## Section 5 &middot; Async and await

Everything so far has been **synchronous**: Python runs one line, waits for it
to finish, then moves to the next. Asynchronous code, written with `async` and
`await`, lets a program start a slow task such as a network call and work on
something else while it waits, instead of sitting idle.

> **Heads up.** **In a script** you start the whole thing with `asyncio.run(main())`. **In a notebook** there is already an event loop running, so `asyncio.run()` raises an error. Use a bare `await main()` instead, as the cells below do. This catches almost everyone out once.

In [ ]:
import asyncio


async def greet():
    print("Starting...")
    await asyncio.sleep(0.5)      # pretend this is waiting on a network call
    print("Done!")


await greet()          # in a script this line would be: asyncio.run(greet())

The real payoff shows up when you need several slow calls at once.
`asyncio.gather` runs them together instead of one after another. Watch the
elapsed time: three half-second calls finish in about half a second, not one
and a half.

In [ ]:
import asyncio
import time


async def call_model(prompt):
    await asyncio.sleep(0.5)      # simulates network latency
    return f"response to: {prompt}"


async def main():
    prompts = ["Summarize A", "Summarize B", "Summarize C"]
    return await asyncio.gather(*(call_model(p) for p in prompts))


start = time.time()
results = await main()
elapsed = time.time() - start

for r in results:
    print(r)

print(f"\nthree calls finished in {elapsed:.2f}s, not {3 * 0.5:.2f}s")

> `async def` and a generator with `yield` both pause and resume, but for
> different reasons. A generator pauses to hand back the next value in a
> sequence. An async function pauses to let something else run while it waits
> on a slow task. `await` is only valid inside a function defined with
> `async def`.

---

### Your turn 2

Validate a cloud resource request with Pydantic, then run three fake model calls at the same time.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
from pydantic import BaseModel
import asyncio


# TODO: inherit from the Pydantic base class, and give each field a type.
class ResourceRequest(____):
    name: ____
    region: ____
    size: ____


req = ResourceRequest(name="web-01", region="us-east-1", size=2)
print(req)


async def fake_call(n):
    await asyncio.sleep(0.2)
    return f"call {n} done"


# TODO: which asyncio function runs them all together?
results = await asyncio.____(fake_call(1), fake_call(2), fake_call(3))
print(results)

---

# More use cases

The same ideas, applied to situations you will meet in real work. Run each one, then change a value and run it again.

## Use case 1 &middot; AI &middot; dataclass and Pydantic, same bad input

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel, ValidationError


@dataclass
class PlainTurn:
    role: str
    content: str


class CheckedTurn(BaseModel):
    role: str
    content: str


bad = PlainTurn(role=123, content=None)
print("dataclass accepted it:", bad)

try:
    CheckedTurn(role=123, content=None)
except ValidationError as e:
    first = e.errors()[0]
    print()
    print("pydantic refused it:", first["msg"], "on field", first["loc"])

## Use case 2 &middot; AI &middot; Validate what the model sent back

In [ ]:
from pydantic import BaseModel, ValidationError


class Triage(BaseModel):
    severity: int
    service: str
    summary: str


good = '{"severity": 2, "service": "api-gateway", "summary": "502s from two targets"}'
bad = '{"severity": "high", "service": "api-gateway", "summary": "502s"}'

result = Triage.model_validate_json(good)
print("parsed:", result.service, "severity", result.severity)
print("type of severity:", type(result.severity))

try:
    Triage.model_validate_json(bad)
except ValidationError as e:
    print()
    print("the model returned severity as text, which is not usable:")
    print(" ", e.errors()[0]["msg"])

## Use case 3 &middot; AI &middot; A decorator that retries

In [ ]:
def retry(attempts=3):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as e:
                    print(f"  attempt {attempt} failed: {e}")
            raise RuntimeError(f"{func.__name__} failed after {attempts} attempts")
        return wrapper
    return decorator


calls = {"n": 0}


@retry(attempts=4)
def fetch_metrics():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("connection reset")
    return "metrics retrieved"


print(fetch_metrics())
print("it took", calls["n"], "calls")

## Use case 4 &middot; AI &middot; Fan out calls of different lengths

In [ ]:
import asyncio
import time


async def call(name, seconds):
    await asyncio.sleep(seconds)
    return f"{name} took {seconds}s"


start = time.time()

results = await asyncio.gather(
    call("summarise", 0.5),
    call("translate", 0.2),
    call("classify", 0.3),
)

elapsed = time.time() - start

for r in results:
    print(r)

print()
print("one after another would be 1.00s")
print(f"all at once took {elapsed:.2f}s, the slowest one")

---

## Lab: A validated, timed, async chat client

Put the whole module together.

Define a Pydantic model `ChatTurn` with a `role` and a `content`, both strings.
Prove it rejects a turn where `role` is a number.

Write a `ChatClient` class that holds a model name and a list of `ChatTurn`
objects. Give it an `async def ask(self, message)` method that sleeps briefly
to simulate the network, appends both the user turn and the assistant reply to
its history, and returns the reply.

Write your own `@timed` decorator and put it on a `report()` method, so
printing the history also prints how long it took.

Finally, use `asyncio.gather` to ask three questions concurrently, then print
the full history and confirm it holds six turns.

**Done when:**

- [ ] ChatTurn is a Pydantic model and rejects a bad role
- [ ] ask() is an async method and is awaited
- [ ] A hand-written @timed decorator wraps report()
- [ ] asyncio.gather runs the three questions together
- [ ] The final history has six turns

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

The four exercises from the module's practice slide are in the
[README](../README.md#practice-exercises) and repeated on the slide. There are
4 of them. Do them in a scratch cell here or in a `.py` file.

## Module complete

That is the whole course. You can now read and write Python confidently enough to work with real AI SDKs.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*